<!-- NOTEBOOK_METADATA source: "\u26a0\ufe0f Jupyter Notebook" title: "Integrate Alibaba Cloud Model Studio (DashScope) with Langfuse" sidebarTitle: "Alibaba Cloud Model Studio" description: "Guide on integrating Alibaba Cloud Model Studio (DashScope) Qwen models with Langfuse for observability, tracing, and cost tracking." category: "Integrations" -->

# Observability for Alibaba Cloud Model Studio (DashScope) with Langfuse

This cookbook demonstrates how to integrate [Alibaba Cloud Model Studio](https://www.alibabacloud.com/en/product/model-studio) (DashScope) with [Langfuse](https://langfuse.com) for full observability of your Qwen model calls.

DashScope provides an **OpenAI-compatible API endpoint**, which means you can use Langfuse's drop-in replacement for the OpenAI SDK to automatically trace all your calls with zero additional code.

We will cover:
- Basic chat completions with automatic tracing
- Streaming responses
- Embedding model tracing
- `@observe()` decorator for grouping multiple generations
- Multi-model comparison (qwen-plus / qwen-turbo / qwen-max) in a single trace
- Viewing traces in the Langfuse dashboard

> **What is Alibaba Cloud Model Studio (DashScope)?** [Model Studio](https://www.alibabacloud.com/en/product/model-studio) is Alibaba Cloud's AI model service platform, providing access to the Qwen family of large language models (qwen-plus, qwen-turbo, qwen-max) and embedding models via an OpenAI-compatible API.

> **What is Langfuse?** [Langfuse](https://langfuse.com) is an open source LLM engineering platform that helps teams trace API calls, monitor performance, and debug issues in their AI applications.

<!-- STEPS_START -->
## Step 1: Install Dependencies

In [ ]:
%pip install openai langfuse --quiet

## Step 2: Set Up Environment Variables

You need:
- A **DashScope API key** from [Alibaba Cloud Model Studio console](https://bailian.console.alibabacloud.com/) (international) or [DashScope console](https://dashscope.console.aliyun.com/) (China)
- **Langfuse credentials** from [Langfuse Cloud](https://cloud.langfuse.com) or your self-hosted instance

Alibaba Cloud Model Studio is available in multiple regions. Use the corresponding API endpoint:

| Region | Endpoint |
|---|---|
| International (Singapore) | `https://dashscope-intl.aliyuncs.com/compatible-mode/v1` |
| US (Virginia) | `https://dashscope-us.aliyuncs.com/compatible-mode/v1` |
| China (Beijing) | `https://dashscope.aliyuncs.com/compatible-mode/v1` |
| EU (Frankfurt) | `https://{WorkspaceId}.eu-central-1.maas.aliyuncs.com/compatible-mode/v1` |

> **Note:** API keys are region-specific and not interchangeable. Ensure your key matches the endpoint region.

> **EU (Frankfurt) setup:** The Frankfurt endpoint requires your Workspace ID in the URL. Replace `{WorkspaceId}` with your actual Workspace ID from the [Model Studio console](https://bailian.console.alibabacloud.com/).

In [ ]:
import os

# Get keys for your project from the project settings page
# https://cloud.langfuse.com
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
os.environ["LANGFUSE_BASE_URL"] = "https://cloud.langfuse.com" # 🇪🇺 EU region
# os.environ["LANGFUSE_BASE_URL"] = "https://us.cloud.langfuse.com" # 🇺🇸 US region

# Get your API key from Alibaba Cloud Model Studio console
# https://bailian.console.alibabacloud.com/
os.environ["DASHSCOPE_API_KEY"] = "sk-..."

# For EU (Frankfurt) region, set your Workspace ID
# os.environ["DASHSCOPE_WORKSPACE_ID"] = "ws-..."

Verify the Langfuse connection:

In [ ]:
from langfuse import get_client

langfuse = get_client()

if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

## Step 3: OpenAI Drop-in Replacement for DashScope

Since DashScope exposes an OpenAI-compatible endpoint, we use the Langfuse drop-in replacement for the OpenAI SDK. Simply import `openai` from `langfuse.openai` and point the `base_url` to DashScope's endpoint.

All API calls are then automatically traced by Langfuse.

In [ ]:
# Instead of: from openai import OpenAI
from langfuse.openai import openai

client = openai.OpenAI(
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
    base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
    # For EU (Frankfurt), use:
    # base_url=f"https://{os.environ['DASHSCOPE_WORKSPACE_ID']}.eu-central-1.maas.aliyuncs.com/compatible-mode/v1",
)

### Basic Chat Completion

A simple chat completion request using `qwen-plus`. The trace is automatically captured in Langfuse.

In [ ]:
response = client.chat.completions.create(
    model="qwen-plus",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is Langfuse and how does it help with LLM observability?"},
    ],
    name="qwen-plus-basic",
)

print(response.choices[0].message.content)

## Step 4: Streaming Chat Completion

Streaming works out of the box. Langfuse captures the full streamed response and token usage.

In [ ]:
stream = client.chat.completions.create(
    model="qwen-plus",
    messages=[
        {"role": "system", "content": "You are a creative storyteller."},
        {"role": "user", "content": "Tell me a short story about a cloud that wanted to become a river."},
    ],
    stream=True,
    name="qwen-plus-streaming",
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="")

## Step 5: Embedding Model Tracing

DashScope also provides embedding models. Langfuse traces these calls just like chat completions.

> **Note:** Embedding models (e.g., `text-embedding-v3`) are currently available in the **China (Beijing)** and **International (Singapore)** regions only. They are not available in the US (Virginia) or EU (Frankfurt) regions.

In [ ]:
embedding_response = client.embeddings.create(
    model="text-embedding-v3",
    input=["Langfuse provides open source LLM observability."],
    name="dashscope-embedding",
)

print(f"Embedding dimension: {len(embedding_response.data[0].embedding)}")
print(f"First 5 values: {embedding_response.data[0].embedding[:5]}")

## Step 6: Group Multiple Generations with `@observe()`

Use the `@observe()` decorator to group multiple LLM calls into a single trace. This is useful when your application logic involves chaining several model calls.

In [ ]:
from langfuse import observe

@observe()
def translate_and_summarize(text: str) -> str:
    # Generation 1: Translate to English
    translation = client.chat.completions.create(
        model="qwen-plus",
        messages=[
            {"role": "system", "content": "You are a translator. Translate the following text to English."},
            {"role": "user", "content": text},
        ],
        name="translate",
    ).choices[0].message.content

    # Generation 2: Summarize the translation
    summary = client.chat.completions.create(
        model="qwen-plus",
        messages=[
            {"role": "system", "content": "You are a summarizer. Provide a one-sentence summary."},
            {"role": "user", "content": translation},
        ],
        name="summarize",
    ).choices[0].message.content

    return summary

result = translate_and_summarize(
    "\u4e91\u8ba1\u7b97\u662f\u4e00\u79cd\u901a\u8fc7\u4e92\u8054\u7f51\u63d0\u4f9b\u8ba1\u7b97\u670d\u52a1\u7684\u6280\u672f\uff0c\u5305\u62ec\u670d\u52a1\u5668\u3001\u5b58\u50a8\u3001\u6570\u636e\u5e93\u548c\u7f51\u7edc\u7b49\u8d44\u6e90\u3002"
)
print(result)

## Step 7: Multi-Model Comparison

Compare responses from different Qwen models (qwen-plus, qwen-turbo, qwen-max) in a single trace. This is useful for evaluating model quality, latency, and cost.

In [ ]:
from langfuse import observe

@observe()
def compare_qwen_models(prompt: str) -> dict:
    models = ["qwen-turbo", "qwen-plus", "qwen-max"]
    results = {}

    for model in models:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a helpful assistant. Be concise."},
                {"role": "user", "content": prompt},
            ],
            name=f"{model}-comparison",
            temperature=0.7,
            max_tokens=200,
        )
        results[model] = response.choices[0].message.content

    return results

comparison = compare_qwen_models("Explain what makes a good API design in three bullet points.")

for model, answer in comparison.items():
    print(f"\n--- {model} ---")
    print(answer)

## Step 8: Enhance Tracing (Optional)

You can enrich your DashScope traces with additional context:

- Add [metadata](https://langfuse.com/docs/tracing-features/metadata), [tags](https://langfuse.com/docs/tracing-features/tags), [log levels](https://langfuse.com/docs/tracing-features/log-levels), and [user IDs](https://langfuse.com/docs/tracing-features/users) to traces
- Group traces by [sessions](https://langfuse.com/docs/tracing-features/sessions)
- Use [Langfuse Prompt Management](https://langfuse.com/docs/prompts/get-started) and link prompts to traces
- Add [scores](https://langfuse.com/docs/scores/custom) to traces

Visit the [OpenAI SDK cookbook](https://langfuse.com/guides/cookbook/integration_openai_sdk) to see more examples on passing additional parameters.

In [ ]:
response = client.chat.completions.create(
    model="qwen-plus",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are the benefits of observability in LLM applications?"},
    ],
    name="qwen-plus-enhanced",
    metadata={
        "langfuse_tags": ["dashscope", "qwen-plus", "demo"],
        "langfuse_user_id": "user-123",
        "langfuse_session_id": "session-abc",
        "langfuse_metadata": {"use_case": "observability-demo", "region": "cn"},
    },
)

print(response.choices[0].message.content)

## Step 9: View Traces in Langfuse

After running the examples above, navigate to [Langfuse Cloud](https://cloud.langfuse.com) (or your self-hosted instance) to see detailed traces including:

- Request and response content
- Token usage and latency metrics
- Cost tracking per model
- Nested traces from `@observe()` decorated functions

You can define custom price information via the Langfuse dashboard ([see docs](https://langfuse.com/docs/model-usage-and-cost)) to configure the exact pricing for your DashScope models.
<!-- STEPS_END -->

<!-- MARKDOWN_COMPONENT name: "LearnMore" path: "@/components-mdx/integration-learn-more.mdx" -->